# Autoencoder with ACCESS-OM2

Autoencoder with ACCESS-OM2 Ocean Heat Content Data

This notebook trains an autoencoder on ocean heat content and surface heat flux data from the ACCESS-OM2 model using partial convolutions to handle land-masked regions.

In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: import dependencies and define paths/device for full workflow.
# Outcome: pipeline/data/model/plot modules are loaded and local utils are importable.
# -----------------------------------------------------------------------------
# System and path handling
import sys
from pathlib import Path

# Data handling
import numpy as np
import pandas as pd
import xarray as xr

# PyEarthTools pipeline
import pyearthtools.data as petdata
import pyearthtools.pipeline as petpipe
import pyearthtools.training

# Deep learning
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import lightning as L

# Visualization
import matplotlib.pyplot as plt

# Configure device and random seed
torch.manual_seed(42)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Source data and code directories
userbase = "/g/data/nm47/txs156/"
# Taimoor's repobase:
repobase = "/home/156/txs156/uom/OM2-emulator/"
# Ryan's repobase:
# repobase = "/home/561/rmh561/ML/OM2-emulator/"
# Navid's repobase:
# repobase = "/home/552/nc3020/gdata/OM2-emulator/"

# Add the src directory to the Python path (relative to notebook location)
src_path = Path(repobase) / "src"
sys.path.insert(0, str(src_path))

# Import local OM2 emulator modules
from Data import ACCESS_OHC, build_normalisation, make_fast_dl
from Emulator import AutoEncoder, LightningWrapper, PartialConv2d, UNet
import warnings

In [ ]:
# Setup the normalisation
time_start = '2000-01'
train_end = "2015-03"
val_start = "2015-04"
time_end = '2018-12'
time_interval = '1MS'
norm_variables = ["ocean_heat_content_2d", "total_surface_heat_flx"]
norm_strat = "Spatial_climatology" 
datapath = userbase + "OM2-emulator/data/1deg_ocean_heat_emulator_data.nc"

mask, normalisation = build_normalisation(datapath, norm_strat, norm_variables, time_window = dict(start=time_start, end=time_end, freq=time_interval),\
                                          train_end = train_end, mask = True)
mean = normalisation._initialisation["mean"]
deviation = normalisation._initialisation["deviation"]

In [ ]:
# Set up accessor for the ACCESS data
ACCESS_OHC_accessor = ACCESS_OHC(
    ["area_t", "ocean_heat_content_2d", "total_surface_heat_flx"],
    root=userbase + "OM2-emulator/data/",
)

## Setup PyEarthTools data accessor and pipeline

In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: assemble one PET pipeline mode and load full
# time series (not just one iterator sample).
# -----------------------------------------------------------------------------

pipeline_i = petpipe.Pipeline(
    ACCESS_OHC_accessor,
    petdata.transforms.coordinates.Drop(['geolat_t', 'geolon_t']),
    petdata.transforms.variables.Drop(['area_t']),
    petpipe.operations.xarray.select.SelectDataset(norm_variables),
    petpipe.operations.xarray.Sort(order=norm_variables, strict=True),
    petpipe.operations.xarray.reshape.Dimensions(["time", "latitude", "longitude"]),
    normalisation,
    petdata.transforms.coordinates.Drop(['geolat_t', 'geolon_t']),
    # The latest branch of PET implements Inf --> 0, so I have this here. 
    # But the latest tagged release does not do anything with these kwargs.
    petpipe.operations.xarray.values.FillNan(0, posinf=0, neginf=0),
    petpipe.operations.xarray.conversion.ToNumpy(),
    petpipe.operations.numpy.reshape.Rearrange('c t h w -> t c h w'),
    iterator=petpipe.iterators.DateRange(time_start, time_end, interval="1 month"),
)

In [ ]:
# We define the training-testing split here!
splits = {
    "train_split": petpipe.iterators.DateRange(
        "2000-01", "2015-04", interval="1 month"
    ),
    "valid_split": petpipe.iterators.DateRange(
        "2015-04", "2019-01", interval="1 month"
    ),
}

## Define the model using Lightning and Torch

## Initialize model and prepare data for training

In [ ]:
# Define the "base" Autoencoder model as before
# base_model = AutoEncoder(
#     input_channel_count=len(norm_variables),
#     output_channel_count=len(norm_variables),
# )

base_model = UNet(
    input_channel_count=len(norm_variables),
    output_channel_count=len(norm_variables),
)

In [ ]:
# Use the lightning wrapper to define the Autoencoder and mask 
lightning_model = LightningWrapper(
    model=base_model,
    mask=mask.values,
    lr=1e-4,
)

In [ ]:
# Use PET's PipelineLightningDataModule to define the data that will be loaded
# into the Lightning wrapper
datamodule = pyearthtools.training.data.lightning.PipelineLightningDataModule(
    pipeline_i,
    **splits,
    batch_size=32,
    num_workers=0,
)

In [ ]:
%%time
# PET training's default mode is to do expensive loading of the datamodule.train_dataloader() every Epoch. 
# This function instead loads the datamodule into memory once, moving from PET --> PyTorch. 
# The shuffle = false kwarg is so that the data is not randomised in time, so
# the un-normalisation later will work


# This call creates the datamodule.train_dataloader() and datamodule.val_dataloader() objects
# Based on the ** splits in the above cell
datamodule.setup("fit")

# Now load the training and validation DataLoaders into memory ONCE
fast_train_dl = make_fast_dl(
    datamodule.train_dataloader(),
    batch_size=32,
    shuffle=False,
    drop_last=False,
)

fast_valid_dl = make_fast_dl(
    datamodule.val_dataloader(),
    batch_size=32,
    shuffle=False,
    drop_last=False,
)

In [ ]:
# This cell defines the Lightning Trainer. 
# PET's native training pyearthtools.training.lightning.Train 
# expects the PipelineLightningDataModule, but
# it is too slow for our current set-up to be implemented here. 
# it takes about 5 mins per epoch

# I have a GH issue on this too: https://github.com/ACCESS-Community-Hub/PyEarthTools/issues/265

trainer = L.Trainer(
    max_epochs=200,
    num_sanity_val_steps=0,
    accelerator="gpu",
    devices=1,
    logger=False,
    enable_checkpointing=False,
    enable_model_summary=False,
)

# This is the bit of code which actually does the training over x epochs. 
trainer.fit(
    model=lightning_model,
    train_dataloaders=fast_train_dl,
    val_dataloaders=fast_valid_dl,
)

In [ ]:
# Now we use the trained Autoencoder to predict both the training and validation data
train_preds = trainer.predict(lightning_model, dataloaders=fast_train_dl)
validation_preds = trainer.predict(lightning_model, dataloaders=fast_valid_dl)

In [ ]:
# The PyTorch output comes as a list of batches, so we concatenate them to get the full array
x_train = torch.cat([p["x"] for p in train_preds], dim=0).cpu().numpy()
xhat_train = torch.cat([p["x_hat"] for p in train_preds], dim=0).cpu().numpy()
x_validation = torch.cat([p["x"] for p in validation_preds], dim=0).cpu().numpy()
xhat_validation = torch.cat([p["x_hat"] for p in validation_preds], dim=0).cpu().numpy()

# The PET un-normalisation expects all times in the array, so we re-concatenate 
# the training and validation data
x_all = np.concatenate([x_train, x_validation], axis=0)
xhat_all = np.concatenate([xhat_train, xhat_validation], axis=0)

In [ ]:
# Undo the PET pipeline
actual_data = pipeline_i.undo(x_all)
predicted_data = pipeline_i.undo(xhat_all)

In [ ]:
# Now we compare the actual and prediction, looks good!! 

month_seconds = 30 * 24 * 3600
itime = 10

fig, axes = plt.subplots(
    nrows=2,
    ncols=3,
    figsize=(18, 8),
    constrained_layout=True,
)

# -------------------------
# Data
# -------------------------
ohc_fields = [
    (actual_data - mean).ocean_heat_content_2d.isel(time=itime),
    (predicted_data - mean).ocean_heat_content_2d.isel(time=itime),
    (predicted_data - actual_data).ocean_heat_content_2d.isel(time=itime),
]

hf_fields = [
    ((actual_data - mean) * month_seconds).total_surface_heat_flx.isel(time=itime),
    ((predicted_data - mean) * month_seconds).total_surface_heat_flx.isel(time=itime),
    ((predicted_data - actual_data) * month_seconds).total_surface_heat_flx.isel(time=itime),
]

col_titles = [
    "Input anomaly",
    "Reconstruction anomaly",
    "Reconstruction error",
]

# -------------------------
# Plot settings
# -------------------------
ohc_vmin, ohc_vmax = -1e9, 1e9
hf_vmin, hf_vmax = -1e8, 1e8
cmap = plt.cm.bwr

# -------------------------
# OHC row
# -------------------------
ohc_mappables = []
for j, field in enumerate(ohc_fields):
    p = field.plot(
        ax=axes[0, j],
        vmin=ohc_vmin,
        vmax=ohc_vmax,
        cmap=cmap,
        add_colorbar=False,
    )
    ohc_mappables.append(p)
    axes[0, j].set_title(col_titles[j])
    axes[0, j].set_xlabel("Longitude")
    axes[0, j].set_ylabel("Latitude" if j == 0 else "")

# -------------------------
# Heat flux row
# -------------------------
hf_mappables = []
for j, field in enumerate(hf_fields):
    p = field.plot(
        ax=axes[1, j],
        vmin=hf_vmin,
        vmax=hf_vmax,
        cmap=cmap,
        add_colorbar=False,
    )
    hf_mappables.append(p)
    axes[1, j].set_title(col_titles[j])
    axes[1, j].set_xlabel("Longitude")
    axes[1, j].set_ylabel("Latitude" if j == 0 else "")

# -------------------------
# One colorbar per row
# -------------------------
cbar_ohc = fig.colorbar(
    ohc_mappables[0],
    ax=axes[0, :],
    orientation="vertical",
    shrink=0.85,
    pad=0.02,
)
cbar_ohc.set_label("Ocean heat content anomaly\n / error [J m$^{-2}$]")

cbar_hf = fig.colorbar(
    hf_mappables[0],
    ax=axes[1, :],
    orientation="vertical",
    shrink=0.85,
    pad=0.02,
)
cbar_hf.set_label("Monthly integrated surface heat flux anomaly\n / error [J m$^{-2}$]")

fig.suptitle(
    f"Autoencoder input, reconstruction, and error at time index {itime}",
    fontsize=16,
)

plt.savefig(repobase + 'figures/Example_output.png', dpi=150, bbox_inches='tight')

plt.show()

In [ ]:
# Plot normalised data example:
OHCnorm_truth = x_all[-1, 0, :, :].copy()
SHFnorm_truth = x_all[-1, 1, :, :].copy()
OHCnorm_pred  = xhat_all[-1, 0, :, :].copy()
SHFnorm_pred  = xhat_all[-1, 1, :, :].copy()

# Mask zeros
for arr in [OHCnorm_truth, SHFnorm_truth, OHCnorm_pred, SHFnorm_pred]:
    arr[arr == 0] = np.nan

# Shared colorbar settings
vmin, vmax = -2.5, 2.5
cmap = 'RdBu_r'

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

panels = [
    (axes[0, 0], OHCnorm_truth, 'OHC Truth'),
    (axes[0, 1], OHCnorm_pred,  'OHC Predicted'),
    (axes[1, 0], SHFnorm_truth, 'SHF Truth'),
    (axes[1, 1], SHFnorm_pred,  'SHF Predicted'),
]

for ax, data, title in panels:
    cm = ax.pcolormesh(data, vmin=vmin, vmax=vmax, cmap=cmap)
    ax.set_title(title)
    ax.set_aspect('equal')
    fig.colorbar(cm, ax=ax, orientation='vertical', fraction=0.046, pad=0.04)

plt.suptitle('Normalised OHC & SHF — Truth vs Predicted', fontsize=13)
plt.tight_layout()
plt.savefig(repobase + 'figures/Example_normalised_output.png', dpi=150, bbox_inches='tight')
plt.show()